# Scraping de Données LinkedIn pour le Machine Learning

## Objectifs
- Extraire des données de profils LinkedIn (compétences, expériences, postes)
- Nettoyer et structurer les données
- Analyser les tendances avec des techniques ML

## ⚠️ Avertissement Légal / Éthique
> LinkedIn's Terms of Service (ToS) restreignent le scraping automatisé.  
> Ce notebook est **uniquement à des fins éducatives**.  
> Utilisez l'**API officielle LinkedIn** pour tout usage en production.  
> Documentation officielle : https://developer.linkedin.com/

## Approches couvertes
1. **`linkedin-api`** — bibliothèque Python non-officielle (usage éducatif)
2. **Données simulées** — pour tester les pipelines ML sans authentification
3. **Analyse & Visualisation** — NLP, clustering de compétences


## 1. Installation et Imports

In [ ]:
# Installation des dépendances
!pip install linkedin-api pandas numpy matplotlib seaborn wordcloud scikit-learn -q

In [ ]:
import os
import time
import random
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Imports réussis")

## 2. Méthode 1 — Scraping avec `linkedin-api` (Authentification requise)

Cette bibliothèque utilise les cookies de session LinkedIn via l'authentification email/mot de passe.

In [ ]:
# ============================================================
# CONFIGURATION — Remplacez par vos identifiants LinkedIn
# NE JAMAIS committer vos identifiants dans git !
# Utilisez des variables d'environnement :
#   export LINKEDIN_EMAIL="votre@email.com"
#   export LINKEDIN_PASSWORD="votre_mot_de_passe"
# ============================================================

LINKEDIN_EMAIL = os.environ.get("LINKEDIN_EMAIL", "")
LINKEDIN_PASSWORD = os.environ.get("LINKEDIN_PASSWORD", "")

USE_REAL_API = bool(LINKEDIN_EMAIL and LINKEDIN_PASSWORD)

if USE_REAL_API:
    print(f"✅ Identifiants trouvés pour : {LINKEDIN_EMAIL}")
else:
    print("⚠️  Aucun identifiant trouvé — Mode données simulées activé")
    print("   Définissez LINKEDIN_EMAIL et LINKEDIN_PASSWORD pour utiliser l'API réelle")

In [ ]:
class LinkedInScraper:
    """
    Wrapper autour de linkedin-api avec gestion des erreurs et rate limiting.
    """
    
    def __init__(self, email: str, password: str):
        from linkedin_api import Linkedin
        self.api = Linkedin(email, password)
        print("✅ Connexion LinkedIn réussie")
    
    def get_profile(self, profile_id: str) -> dict:
        """Récupère un profil LinkedIn complet."""
        time.sleep(random.uniform(1.5, 3.0))  # Rate limiting
        try:
            profile = self.api.get_profile(profile_id)
            return self._parse_profile(profile)
        except Exception as e:
            print(f"❌ Erreur profil {profile_id}: {e}")
            return {}
    
    def search_people(self, keywords: str, limit: int = 20) -> list:
        """Recherche des personnes par mots-clés."""
        time.sleep(random.uniform(2.0, 4.0))
        try:
            results = self.api.search_people(
                keywords=keywords,
                limit=limit
            )
            return results
        except Exception as e:
            print(f"❌ Erreur recherche: {e}")
            return []
    
    def get_job_postings(self, keywords: str, location: str = "", limit: int = 50) -> list:
        """Récupère des offres d'emploi."""
        time.sleep(random.uniform(1.5, 3.0))
        try:
            jobs = self.api.search_jobs(
                keywords=keywords,
                location_name=location,
                limit=limit
            )
            return [self._parse_job(j) for j in jobs]
        except Exception as e:
            print(f"❌ Erreur offres emploi: {e}")
            return []
    
    def _parse_profile(self, raw: dict) -> dict:
        """Normalise les données d'un profil."""
        experiences = raw.get('experience', [])
        education = raw.get('education', [])
        skills = raw.get('skills', [])
        
        return {
            'id': raw.get('profile_id', ''),
            'nom': raw.get('lastName', ''),
            'prenom': raw.get('firstName', ''),
            'titre': raw.get('headline', ''),
            'localisation': raw.get('locationName', ''),
            'connections': raw.get('connections', 0),
            'nb_experiences': len(experiences),
            'nb_formations': len(education),
            'competences': [s.get('name', '') for s in skills],
            'nb_competences': len(skills),
            'entreprise_actuelle': experiences[0].get('companyName', '') if experiences else '',
            'poste_actuel': experiences[0].get('title', '') if experiences else '',
            'derniere_formation': education[0].get('schoolName', '') if education else '',
            'diplome': education[0].get('degreeName', '') if education else '',
        }
    
    def _parse_job(self, raw: dict) -> dict:
        """Normalise une offre d'emploi."""
        return {
            'id': raw.get('entityUrn', '').split(':')[-1],
            'titre': raw.get('title', ''),
            'entreprise': raw.get('companyName', ''),
            'localisation': raw.get('formattedLocation', ''),
            'type_contrat': raw.get('employmentType', ''),
            'date_publication': raw.get('listedAt', ''),
        }


# Initialisation conditionnelle
scraper = None
if USE_REAL_API:
    try:
        scraper = LinkedInScraper(LINKEDIN_EMAIL, LINKEDIN_PASSWORD)
    except Exception as e:
        print(f"❌ Connexion échouée: {e}")
        USE_REAL_API = False

## 3. Méthode 2 — Données Simulées (Mode Démo / Test ML Pipeline)

In [ ]:
def generate_linkedin_data(n_profiles: int = 200) -> pd.DataFrame:
    """
    Génère un dataset réaliste de profils LinkedIn simulés.
    Utile pour tester les pipelines ML sans authentification.
    """
    np.random.seed(42)
    
    # Données réalistes
    titres = [
        'Data Scientist', 'Machine Learning Engineer', 'Data Analyst',
        'AI Research Scientist', 'MLOps Engineer', 'Data Engineer',
        'Business Intelligence Analyst', 'NLP Engineer', 'Computer Vision Engineer',
        'Quantitative Analyst', 'Deep Learning Researcher', 'AI Product Manager'
    ]
    
    entreprises = [
        'Google', 'Meta', 'Amazon', 'Microsoft', 'Apple', 'Netflix',
        'Airbus', 'BNP Paribas', 'Capgemini', 'Thales', 'Orange', 'Société Générale',
        'Startup IA', 'Freelance', 'SNCF', 'EDF', 'Danone', 'TotalEnergies'
    ]
    
    formations = [
        'Polytechnique', 'CentraleSupélec', 'ENSAE', 'Télécom Paris',
        'HEC Paris', 'Sorbonne Université', 'Université Paris-Saclay',
        'MIT', 'Stanford', 'ETH Zurich', 'EPFL', 'Mines ParisTech'
    ]
    
    diplomes = ['Master', 'PhD', 'MBA', 'Ingénieur', 'Bachelor', 'MSc']
    localisations = ['Paris', 'Lyon', 'Toulouse', 'Bordeaux', 'Marseille', 'Lille', 
                     'Remote', 'London', 'Berlin', 'Amsterdam', 'Zurich']
    
    competences_pool = [
        'Python', 'R', 'SQL', 'TensorFlow', 'PyTorch', 'Scikit-learn',
        'Spark', 'Hadoop', 'Kafka', 'Docker', 'Kubernetes', 'AWS', 'GCP', 'Azure',
        'NLP', 'Computer Vision', 'Deep Learning', 'Reinforcement Learning',
        'Statistics', 'A/B Testing', 'Data Visualization', 'Tableau', 'Power BI',
        'Git', 'MLflow', 'Airflow', 'FastAPI', 'Streamlit', 'Pandas', 'NumPy'
    ]
    
    records = []
    for i in range(n_profiles):
        titre = np.random.choice(titres)
        n_skills = np.random.randint(5, 20)
        skills = list(np.random.choice(competences_pool, n_skills, replace=False))
        
        records.append({
            'id': f'profile_{i+1:04d}',
            'titre': titre,
            'entreprise_actuelle': np.random.choice(entreprises),
            'localisation': np.random.choice(localisations),
            'derniere_formation': np.random.choice(formations),
            'diplome': np.random.choice(diplomes),
            'nb_experiences': np.random.randint(1, 12),
            'nb_formations': np.random.randint(1, 4),
            'connections': int(np.random.lognormal(6.5, 1.0)),
            'nb_competences': n_skills,
            'competences': ', '.join(skills),
            'annees_experience': np.random.randint(0, 20),
            'salaire_estime_keur': np.random.normal(
                loc=65 if 'Senior' not in titre else 90,
                scale=15
            ).clip(30, 180),
        })
    
    return pd.DataFrame(records)


# Chargement des données
if USE_REAL_API and scraper:
    print("📡 Collecte via API LinkedIn...")
    results = scraper.search_people("Data Scientist", limit=50)
    profiles = [scraper.get_profile(r['urn_id']) for r in results[:20]]
    df = pd.DataFrame([p for p in profiles if p])
    print(f"✅ {len(df)} profils récupérés via API")
else:
    print("🔧 Génération de données simulées...")
    df = generate_linkedin_data(n_profiles=300)
    print(f"✅ Dataset simulé : {len(df)} profils générés")

df.head()

## 4. Exploration et Nettoyage des Données

In [ ]:
print("=" * 50)
print("APERÇU DU DATASET")
print("=" * 50)
print(f"\n📊 Dimensions : {df.shape[0]} profils × {df.shape[1]} colonnes")
print(f"\n📋 Colonnes disponibles :")
for col in df.columns:
    print(f"  • {col} ({df[col].dtype})")

print(f"\n🔍 Valeurs manquantes :")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "  Aucune valeur manquante")

print(f"\n📈 Statistiques descriptives :")
df.describe().round(2)

## 5. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Analyse des Profils LinkedIn — Data Science & IA', fontsize=16, fontweight='bold')

# 1. Distribution des titres
titre_counts = df['titre'].value_counts().head(10)
axes[0, 0].barh(titre_counts.index, titre_counts.values, color='steelblue')
axes[0, 0].set_title('Top 10 Titres de Poste')
axes[0, 0].set_xlabel('Nombre de profils')

# 2. Distribution des localisations
loc_counts = df['localisation'].value_counts().head(8)
axes[0, 1].pie(loc_counts.values, labels=loc_counts.index, autopct='%1.1f%%',
               colors=sns.color_palette('pastel'))
axes[0, 1].set_title('Répartition Géographique')

# 3. Distribution des connexions
axes[0, 2].hist(df['connections'].clip(0, 5000), bins=30, color='salmon', edgecolor='white')
axes[0, 2].set_title('Distribution des Connexions')
axes[0, 2].set_xlabel('Nombre de connexions')
axes[0, 2].set_ylabel('Fréquence')

# 4. Expériences vs Compétences
axes[1, 0].scatter(df['nb_experiences'], df['nb_competences'],
                   alpha=0.5, c=df['annees_experience'],
                   cmap='viridis', s=50)
axes[1, 0].set_title('Expériences vs Compétences')
axes[1, 0].set_xlabel('Nb expériences')
axes[1, 0].set_ylabel('Nb compétences')

# 5. Distribution des diplômes
diplome_counts = df['diplome'].value_counts()
axes[1, 1].bar(diplome_counts.index, diplome_counts.values,
               color=sns.color_palette('husl', len(diplome_counts)))
axes[1, 1].set_title('Types de Diplômes')
axes[1, 1].set_xlabel('Diplôme')
axes[1, 1].set_ylabel('Nombre')
axes[1, 1].tick_params(axis='x', rotation=30)

# 6. Distribution des années d'expérience
axes[1, 2].hist(df['annees_experience'], bins=20, color='mediumseagreen', edgecolor='white')
axes[1, 2].set_title("Distribution des Années d'Expérience")
axes[1, 2].set_xlabel("Années d'expérience")
axes[1, 2].set_ylabel('Fréquence')

plt.tight_layout()
plt.savefig('linkedin_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Nuage de mots des compétences
all_skills = ' '.join(df['competences'].dropna().tolist())

wc = WordCloud(
    width=1200, height=600,
    background_color='white',
    colormap='Blues',
    max_words=80,
    collocations=False
).generate(all_skills)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Compétences les plus fréquentes sur LinkedIn (Data Science)', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('linkedin_skills_wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 20 compétences
skills_list = [s.strip() for skills in df['competences'].dropna() for s in skills.split(',')]
top_skills = pd.Series(Counter(skills_list)).sort_values(ascending=False).head(20)

plt.figure(figsize=(12, 6))
bars = plt.barh(top_skills.index[::-1], top_skills.values[::-1],
                color=plt.cm.Blues(np.linspace(0.4, 0.9, 20)))
plt.title('Top 20 Compétences — Profils Data Science LinkedIn', fontsize=13)
plt.xlabel('Nombre de profils')
for bar, val in zip(bars, top_skills.values[::-1]):
    plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('linkedin_top_skills.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Machine Learning — Clustering de Profils

On applique **TF-IDF + K-Means** pour regrouper les profils selon leurs compétences.

In [ ]:
# --- TF-IDF sur les compétences ---
tfidf = TfidfVectorizer(
    tokenizer=lambda x: [s.strip() for s in x.split(',')],
    token_pattern=None,
    lowercase=True
)

X_tfidf = tfidf.fit_transform(df['competences'].fillna(''))
print(f"Matrice TF-IDF : {X_tfidf.shape} (profils × termes)")

# --- Méthode du coude pour choisir k ---
inertias = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_tfidf)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Nombre de clusters k')
plt.ylabel('Inertie')
plt.title('Méthode du Coude — Clustering des Profils LinkedIn')
plt.xticks(K_range)
plt.tight_layout()
plt.show()

In [ ]:
# --- Clustering final avec k=4 ---
K_OPTIMAL = 4

kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_tfidf)

print(f"Répartition des {K_OPTIMAL} clusters :")
print(df['cluster'].value_counts().to_frame('nb_profils').rename_axis('cluster'))

# Caractérisation de chaque cluster
feature_names = tfidf.get_feature_names_out()
print("\n--- Compétences caractéristiques par cluster ---")
for i in range(K_OPTIMAL):
    top_terms_idx = kmeans.cluster_centers_[i].argsort()[-8:][::-1]
    top_terms = [feature_names[j] for j in top_terms_idx]
    cluster_titles = df[df['cluster'] == i]['titre'].value_counts().head(2).index.tolist()
    print(f"\n🔵 Cluster {i} ({len(df[df['cluster']==i])} profils)")
    print(f"   Compétences : {', '.join(top_terms)}")
    print(f"   Postes fréquents : {', '.join(cluster_titles)}")

In [ ]:
# --- Visualisation 2D avec PCA ---
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_tfidf.toarray())

plt.figure(figsize=(10, 7))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
cluster_names = [
    'Cluster 0 — Engineering / Cloud',
    'Cluster 1 — Data Science / Stats',
    'Cluster 2 — Deep Learning / Vision',
    'Cluster 3 — BI / Analytics'
]

for c in range(K_OPTIMAL):
    mask = df['cluster'] == c
    plt.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=colors[c], label=cluster_names[c],
        alpha=0.6, s=60, edgecolors='white', linewidth=0.5
    )

plt.title('Clustering K-Means des Profils LinkedIn (PCA 2D)', fontsize=13)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('linkedin_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export des Données

In [ ]:
# Export CSV
output_file = 'linkedin_profiles_data.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"✅ Données exportées vers : {output_file}")
print(f"   Taille : {len(df)} profils × {len(df.columns)} colonnes")

# Export JSON
df.to_json('linkedin_profiles_data.json', orient='records', force_ascii=False, indent=2)
print("✅ Données exportées au format JSON")

# Résumé statistique
print("\n📊 Résumé du dataset final :")
print(df.describe(include='all').T[['count', 'unique', 'top', 'mean']].fillna(''))

## 8. Récapitulatif

| Étape | Description |
|-------|-------------|
| **Collecte** | `linkedin-api` (authentifié) ou données simulées |
| **Nettoyage** | Normalisation, valeurs manquantes |
| **Exploration** | Distributions, top compétences, géographie |
| **NLP** | TF-IDF sur les compétences |
| **ML** | K-Means clustering + visualisation PCA |
| **Export** | CSV + JSON |

### Pour aller plus loin
- Prédiction du salaire avec **régression** (XGBoost, RandomForest)
- Classification du niveau séniorité avec **Random Forest**
- Analyse de sentiment des descriptions avec **BERT**
- Recommandation de profils similaires avec **cosine similarity**
- API officielle LinkedIn : https://developer.linkedin.com/docs/v2
